# ETS Machine Learning II — Prediksi RNFL 12 Clock-Hour dari Fundus

**Tujuan:** memperkirakan ketebalan RNFL 12 sektor dari foto fundus untuk membantu analisis saraf optik. Satuan target dan metrik adalah µm. Ini proyek eksperimen, bukan alat diagnosis klinis.

Notebook ini meneruskan replikasi Chen et al. dengan empat komponen sesuai rubrik ETS: **akuisisi/wrangling data**, **augmentasi citra**, **ensemble learning**, dan **reinforcement learning berbentuk contextual-free multi-armed bandit** untuk memilih strategi akuisisi data berlabel. Evaluasi utama memakai 5-fold GroupKFold pasien. Validasi reward bandit dipisah dari fold luar sehingga label fold luar tidak dipakai untuk memilih data.

Baseline Chen: OLS, Ridge, Lasso, ElasticNet. Model ETS membandingkan Ridge, ExtraTrees, gabungan rata-rata keduanya, dan ensemble dengan augmentasi fotometrik. Kurva akuisisi membandingkan kebijakan bandit UCB1 dengan pemilihan pasien acak pada anggaran label yang sama.

Paper acuan: Chen et al., *Independent Evaluation of RETFound Foundation Model’s Performance on Optic Nerve Analysis Using Fundus Photography*, Ophthalmology Science (2025), DOI: https://doi.org/10.1016/j.xops.2025.100720.

**Batas pairing:** tanggal foto tidak tersedia; kriteria selang 6 bulan dari paper belum dapat diverifikasi. Audit pasangan dan jumlah data yang tersisih diekspor.

## 1. Input dan cara menjalankan di Kaggle

1. Buat **Kaggle Dataset** privat yang berisi `rnfl_metadata_lengkap.csv` dan folder `fundus/` dengan foto sesuai metadata. Tambahkan dataset itu melalui **Add Input** di notebook Kaggle. Tidak perlu menaruh CSV atau foto di `/kaggle/working`.
2. Opsional: tambahkan laporan OCT sesuai `oct_filename` untuk QC, dan `fundus_oct_pairs_reviewed.csv` bila pasangan foto telah diperiksa manual.
3. Opsional untuk replikasi Chen: tambahkan dataset berisi kode resmi `RETFound_official/models_vit.py` bersama folder `util/`, checkpoint **`RETFound_mae_natureCFP.pth`**, dan paket `timm` yang kompatibel. Aktifkan GPU untuk ekstraksi RETFound. [Kode resmi](https://github.com/rmaphoh/RETFound) dan [checkpoint CFP](https://huggingface.co/YukunZhou/RETFound_mae_natureCFP) harus diunggah sebagai Kaggle Input sesuai izin penggunaannya. Checkpoint OCT tidak cocok untuk foto fundus.
4. Jalankan **Run All**. Hasil CSV dan fitur tersimpan di `/kaggle/working/rnfl_ets_outputs/` dan dapat diunduh dari tab Output. Internet Kaggle tidak diperlukan setelah semua input terpasang.

**Mode tanpa RETFound:** notebook memakai fitur gambar deterministik (thumbnail abu-abu, histogram warna, statistik warna), sehingga pipeline ETS tetap dapat dijalankan hanya dengan CSV dan foto. Angka pada mode ini adalah baseline eksploratif, **bukan** hasil replikasi RETFound/Chen. `feature_source` dicatat pada output.

Foto fundus dipertahankan orientasinya. Augmentasi hanya mengubah kecerahan, kontras, dan saturasi secara ringan. Citra validasi dan target RNFL tidak diaugmentasi. Transformasi kedua dihitung untuk seluruh citra demi cache fitur, tetapi hanya baris **train** dari tiap fold yang dipakai untuk fit model.

In [ ]:
from pathlib import Path
import os, re, sys, json, math, warnings, random
import numpy as np
try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms
    from torchvision.transforms import InterpolationMode
except (ImportError, RuntimeError) as exc:
    torch = None
    print("PyTorch/torchvision tidak tersedia; fitur gambar sederhana akan dipakai:", exc)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageEnhance
from IPython.display import display

SEED = 42
N_FOLDS = 5
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd() / "input"
OUTPUT_DIR = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()) / "rnfl_ets_outputs"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") if torch else None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
sns.set_theme(style="whitegrid", context="notebook")

image_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
csv_hits = sorted(INPUT_ROOT.rglob("rnfl_metadata_lengkap.csv"))
if not csv_hits:
    raise FileNotFoundError(f"Tidak menemukan rnfl_metadata_lengkap.csv di {INPUT_ROOT}. Tambahkan Kaggle Input sesuai bagian 1.")
CSV_PATH = csv_hits[0]
fundus_hits = []
for p in INPUT_ROOT.rglob("fundus"):
    if p.is_dir() and any(x.is_file() and x.suffix.lower() in image_exts for x in p.rglob("*")):
        fundus_hits.append(p)
if not fundus_hits:
    raise FileNotFoundError(f"Folder fundus berisi citra tidak ditemukan di {INPUT_ROOT}.")
FUNDUS_DIR = sorted(fundus_hits, key=lambda p: len(p.parts))[0]
oct_image_paths = [p for p in INPUT_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in image_exts and p.name.lower().startswith("oct_")]
oct_image_index = {}
for p in sorted(oct_image_paths): oct_image_index.setdefault(p.name.lower(), []).append(p)
PAIR_OVERRIDE_HITS = sorted(INPUT_ROOT.rglob("fundus_oct_pairs_reviewed.csv"))
PAIR_OVERRIDE_PATH = PAIR_OVERRIDE_HITS[0] if PAIR_OVERRIDE_HITS else None
MODEL_CODE_HITS = sorted(INPUT_ROOT.rglob("models_vit.py"))
WEIGHT_HITS = sorted(p for p in INPUT_ROOT.rglob("*.pth") if "retfound" in p.name.lower() and ("cfp" in p.name.lower() or "fundus" in p.name.lower()))
print("Metadata:", CSV_PATH, "| Fundus:", FUNDUS_DIR)
print("OCT:", len(oct_image_paths), "| Pairing manual:", PAIR_OVERRIDE_PATH)
print("Kode RETFound:", MODEL_CODE_HITS[:2], "| Bobot CFP:", WEIGHT_HITS[:2])
print("Output:", OUTPUT_DIR, "| Device:", DEVICE)

Metadata: /kaggle/input/datasets/septiadiniarifta/rnfl-fundus-sectors/rnfl_metadata_lengkap.csv | Fundus: /kaggle/input/datasets/septiadiniarifta/rnfl-fundus-sectors/fundus
OCT: 0 | Pairing manual: None
Kode RETFound: [PosixPath('/kaggle/input/datasets/septiadiniarifta/retfound/RETFound_official/models_vit.py')] | Bobot CFP: [PosixPath('/kaggle/input/datasets/septiadiniarifta/retfound/RETFound_official/RETFound_mae_natureCFP.pth')]
Output: /kaggle/working/rnfl_ets_outputs | Device: cuda
